In [7]:
# Importações e ínicio da sessão spark

from pyspark.sql import SparkSession
from etl_utils import aggregate_genre_trends
import os

spark = SparkSession.builder.appName("MovieETL_Load").getOrCreate()

print("Bibliotecas e funções importadas, Spark iniciado")

Bibliotecas e funções importadas, Spark iniciado


In [8]:
# Carregar da camada silver

base_path = "../input"
silver_path = os.path.join(base_path, "silver")
gold_path = os.path.join(base_path, "gold")

os.makedirs(gold_path, exist_ok=True)

enriched_df = spark.read.parquet(os.path.join(silver_path, "enriched.parquet"))

print("Diretórios setados e criados e parquet lido")

Diretórios setados e criados e parquet lido


In [9]:
# Agregar Trends

trends_df = aggregate_genre_trends(enriched_df)

trends_df.write.mode("overwrite").parquet(os.path.join(gold_path, "genre_trends.parquet"))
trends_df.show(20)  # Preview

print(f"Foram carregados genre trends para gold. Total de linhas: {trends_df.count()}")

+----------+-----------+-----------+----------+--------------+----------+-----------+
|movie_year|      genre|movie_count|avg_rating|avg_popularity|avg_budget|avg_revenue|
+----------+-----------+-----------+----------+--------------+----------+-----------+
|      2000|     Action|          1|       1.4|          NULL|      NULL|       NULL|
|      2000|     Comedy|          5|      4.83|          NULL|      NULL|       NULL|
|      2000|      Crime|          2|       3.8|          NULL|      NULL|       NULL|
|      2000|Documentary|          1|       7.0|          NULL|      NULL|       NULL|
|      2000|      Drama|          1|       5.7|          NULL|      NULL|       NULL|
|      2000|     Family|          1|       7.8|          NULL|      NULL|       NULL|
|      2000|    Romance|          3|      7.17|          NULL|      NULL|       NULL|
|      2000|      Sport|          1|       6.2|          NULL|      NULL|       NULL|
|      2001|     Action|          1|       5.9|       

In [10]:
# Exportar genre_trends também como CSV formatado

csv_output_path = os.path.join(gold_path, "genre_trends.csv")

(
    trends_df
    .coalesce(1) 
    .write
    .mode("overwrite")
    .option("header", True)
    .option("sep", ";") 
    .option("quote", '"')
    .csv(csv_output_path)
)

print(f"Arquivo CSV exportado com sucesso em: {csv_output_path}")


Arquivo CSV exportado com sucesso em: ../input\gold\genre_trends.csv


In [5]:
# Parando o spark

spark.stop()